# v2 — Cross-Encoder 문장 재순위 및 판정 실험

이 파일은 실험 기록을 해당 버전 시점까지 포함한 누적 Colab 노트북입니다.
실행 출력과 학습 가중치는 포함하지 않습니다. 위에서부터 순서대로 실행하세요.


In [ ]:
%cd /content
!git clone https://github.com/hover-nlp/hover.git
%cd /content/hover
!bash download_data.sh

In [ ]:
import json
import sqlite3
from pathlib import Path

base = Path("/content/hover")

train_path = base / "data/hover/hover_train_release_v1.1.json"
retrieval_path = base / "data/hover/tfidf_retrieved/train_tfidf_doc_retrieval_results.json"
db_path = base / "data/wiki_wo_links.db"

for path in [train_path, retrieval_path, db_path]:
    print(path, "→", "있음" if path.exists() else "없음")

with open(train_path, "r", encoding="utf-8") as f:
    train_raw = json.load(f)

with open(retrieval_path, "r", encoding="utf-8") as f:
    retrieval_raw = json.load(f)

conn = sqlite3.connect(str(db_path))
cursor = conn.cursor()

print("\nTrain 자료형:", type(train_raw), "| 샘플 수:", len(train_raw))
print("Train 첫 샘플 필드:", list(train_raw[0].keys()))
print("Train 첫 샘플:", train_raw[0])

print("\n검색 결과 자료형:", type(retrieval_raw), "| 샘플 수:", len(retrieval_raw))
print("검색 결과 첫 샘플 필드:", list(retrieval_raw[0].keys()))

print("\nDB 테이블:", cursor.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall())

In [ ]:
from collections import Counter
from sklearn.model_selection import train_test_split

# 2-hop 샘플만 선택
two_hop_data = [
    row for row in train_raw
    if row["num_hops"] == 2
]

# 기존과 같은 80:20 분할, 라벨 비율 유지
train_data, val_data = train_test_split(
    two_hop_data,
    test_size=0.2,
    random_state=42,
    stratify=[row["label"] for row in two_hop_data]
)

# claim으로 사전 검색 결과를 찾을 수 있도록 연결
retrieval_by_claim = {
    row["claim"]: row
    for row in retrieval_raw
}

print("2-hop 전체:", len(two_hop_data))
print("학습:", len(train_data), Counter(row["label"] for row in train_data))
print("검증:", len(val_data), Counter(row["label"] for row in val_data))

train_found = sum(row["claim"] in retrieval_by_claim for row in train_data)
val_found = sum(row["claim"] in retrieval_by_claim for row in val_data)

print(f"학습 claim 검색 결과 연결: {train_found}/{len(train_data)}")
print(f"검증 claim 검색 결과 연결: {val_found}/{len(val_data)}")

In [ ]:
import inspect

for name, obj in list(globals().items()):
    if inspect.isfunction(obj) and any(
        word in name.lower()
        for word in ["predict", "inference", "verify", "two_hop", "fact"]
    ):
        try:
            print(name, inspect.signature(obj))
        except Exception:
            print(name)

In [ ]:
import inspect

for name, obj in list(globals().items()):
    if inspect.isfunction(obj) and obj.__module__ == "__main__":
        try:
            print(f"{name}{inspect.signature(obj)}")
        except Exception:
            print(name)

In [ ]:
for name, value in list(globals().items()):
    if any(word in name.lower() for word in [
        "model", "verifier", "cross", "rank", "retriev",
        "claim", "evidence", "two_hop", "wiki", "db", "conn"
    ]):
        print(f"{name}: {type(value).__name__}")

In [ ]:
from pathlib import Path

for path in Path("/content").iterdir():
    print(path.name, "— 폴더" if path.is_dir() else "— 파일")

In [ ]:
from pathlib import Path

for path in Path("/content/hover").iterdir():
    print(path.name, "— 폴더" if path.is_dir() else f"— 파일 ({path.stat().st_size:,} bytes)")

In [ ]:
from pathlib import Path

for root in [Path("/content/hover"), Path("/content")]:
    for path in root.rglob("*"):
        if path.is_file() and path.name.lower() in {
            "config.json", "model.safetensors", "pytorch_model.bin",
            "model.safetensors.index.json"
        }:
            print(path)

In [ ]:
from pathlib import Path

model_path = Path("/content/hover_final_verifier_v5")

print("모델 체크포인트 존재:", model_path.exists())
if model_path.exists():
    print("저장된 파일:", [p.name for p in model_path.iterdir()])
else:
    print("현재 런타임에는 체크포인트가 없습니다.")

In [ ]:
!pip -q install blingfire

from functools import lru_cache
from blingfire import text_to_sentences

@lru_cache(maxsize=2000)
def get_sentences(title):
    result = conn.execute(
        "SELECT text FROM documents WHERE id = ?",
        (title,)
    ).fetchone()

    if result is None:
        return ()

    sentences = text_to_sentences(result[0]).splitlines()
    return tuple(sentence.strip() for sentence in sentences if sentence.strip())


# 학습 데이터의 첫 샘플만 확인
sample = train_data[0]

print("Claim:", sample["claim"])
print("라벨:", sample["label"])

for title, sentence_id in sample["supporting_facts"]:
    sentences = get_sentences(title)

    print(f"\n문서: {title}")
    print(f"정답 문장 번호: {sentence_id}")
    if sentence_id < len(sentences):
        print("정답 근거 문장:", sentences[sentence_id])
    else:
        print("DB에서 해당 문장 번호를 찾지 못함. 문장 분리 방식 확인이 필요함.")

In [ ]:
output_path="/content/drive/MyDrive/hover_final_verifier_v5"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
print("학습된 v5 모델:", "verifier_v5" in globals())
print("v5 학습 데이터:", "train_loader_v5" in globals())

In [ ]:
from pathlib import Path

paths = [
    "/content/hover/data/hover/hover_train_release_v1.1.json",
    "/content/hover/data/hover/tfidf_retrieved/train_tfidf_doc_retrieval_results.json",
    "/content/hover/data/wiki_wo_links.db",
]

for path in paths:
    p = Path(path)
    print("있음" if p.exists() else "없음", path)

In [ ]:
import json
import sqlite3
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

base = Path("/content/hover")

with open(
    base / "data/hover/hover_train_release_v1.1.json",
    encoding="utf-8"
) as f:
    train_raw = json.load(f)

with open(
    base / "data/hover/tfidf_retrieved/train_tfidf_doc_retrieval_results.json",
    encoding="utf-8"
) as f:
    retrieval_raw = json.load(f)

conn = sqlite3.connect(str(base / "data/wiki_wo_links.db"))

two_hop_data = [
    row for row in train_raw
    if row["num_hops"] == 2
]

train_data, val_data = train_test_split(
    two_hop_data,
    test_size=0.2,
    random_state=42,
    stratify=[row["label"] for row in two_hop_data]
)

retrieval_by_claim = {
    row["claim"]: row
    for row in retrieval_raw
}

print("2-hop 전체:", len(two_hop_data))
print("학습:", len(train_data), Counter(row["label"] for row in train_data))
print("검증:", len(val_data), Counter(row["label"] for row in val_data))
print("DB 연결:", conn.execute("SELECT COUNT(*) FROM documents").fetchone()[0])

In [ ]:
retrieval_by_claim = {
    row["claim"]: row
    for row in retrieval_raw
}

titles_by_claim_top50 = {}

for claim, row in retrieval_by_claim.items():
    doc_results = row["doc_retrieval_results"]

    # 검색 결과 구조에서 문서 제목 목록을 꺼냄
    titles = doc_results[0][0]

    titles_by_claim_top50[claim] = titles[:50]

train_found = sum(
    row["claim"] in titles_by_claim_top50
    for row in train_data
)
val_found = sum(
    row["claim"] in titles_by_claim_top50
    for row in val_data
)

print(f"학습 claim 연결: {train_found}/{len(train_data)}")
print(f"검증 claim 연결: {val_found}/{len(val_data)}")

sample_claim = train_data[0]["claim"]
print("\n예시 claim:", sample_claim)
print("Top-5 문서:", titles_by_claim_top50[sample_claim][:5])

In [ ]:
from pathlib import Path
import importlib.util

corenlp_dir = Path("/content/corenlp-unpacked/stanford-corenlp-4.6.0")

print("CoreNLP 폴더:", corenlp_dir.exists())
print("CoreNLP jar 파일 수:", len(list(corenlp_dir.glob("*.jar"))))
print("spaCy 설치:", importlib.util.find_spec("spacy") is not None)

if importlib.util.find_spec("spacy"):
    import spacy
    print("spaCy 영어 모델 설치:", spacy.util.is_package("en_core_web_sm"))

In [ ]:
from tqdm.auto import tqdm

positive_pairs = []
missing_titles = []
bad_sentence_indices = []

for row in tqdm(train_data):
    for title, sentence_id in row["supporting_facts"]:
        sentences = get_sentences(title)

        if not sentences:
            missing_titles.append(title)
            continue

        if sentence_id >= len(sentences):
            bad_sentence_indices.append((title, sentence_id, len(sentences)))
            continue

        positive_pairs.append({
            "claim": row["claim"],
            "sentence": sentences[sentence_id],
            "title": title,
            "sentence_id": sentence_id,
            "relevant": 1
        })

print("정답 근거 문장 예시 수:", len(positive_pairs))
print("DB에서 문서를 못 찾은 수:", len(missing_titles))
print("문장 번호를 찾지 못한 수:", len(bad_sentence_indices))

print("\n예시:")
print("Claim:", positive_pairs[0]["claim"])
print("근거 문장:", positive_pairs[0]["sentence"])
print("검색 학습 라벨 relevant:", positive_pairs[0]["relevant"])

In [ ]:
from collections import Counter

print("누락된 서로 다른 문서 제목 수:", len(set(missing_titles)))
print("누락 제목 예시:")
for title, count in Counter(missing_titles).most_common(10):
    print(repr(title), "| 횟수:", count)

print("\n범위를 벗어난 문장 번호 예시:")
for title, sentence_id, sentence_count in bad_sentence_indices[:10]:
    print(
        "문서:", repr(title),
        "| 요청 번호:", sentence_id,
        "| 분리된 문장 수:", sentence_count
    )

In [ ]:
# 제목이 정확히 일치하지 않는 사례를 DB에서 비슷한 제목으로 찾아보기
for title in ["Matthieu Vaxivière", "Renée Montagne", "Émile Driant"]:
    print(f"\n요청 제목: {title!r}")

    # 앞부분으로 시작하는 문서 제목을 확인합니다.
    prefix = title.split()[0] + "%"
    matches = conn.execute(
        "SELECT id FROM documents WHERE id LIKE ? LIMIT 10",
        (prefix,)
    ).fetchall()

    print("DB 제목 후보:", [row[0] for row in matches])

In [ ]:
import unicodedata

def title_key(text):
    # 악센트를 제거하고 소문자로 바꿔 제목 표기를 비교합니다.
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text.casefold()

for requested in [
    "Matthieu Vaxivière",
    "Renée Montagne",
    "Émile Driant",
]:
    first_word = requested.split()[0]
    # 첫 단어의 악센트 표기 차이도 고려해 후보 제목을 가져옵니다.
    prefix = title_key(first_word) + "%"

    candidates = conn.execute(
        "SELECT id FROM documents WHERE id LIKE ? LIMIT 5000",
        (prefix,)
    ).fetchall()

    matches = [
        title for (title,) in candidates
        if title_key(title).startswith(title_key(requested))
    ]

    print(f"\n요청 제목: {requested!r}")
    print("악센트 무시 일치 제목:", matches[:10])
    print("검색한 후보 제목 수:", len(candidates))

In [ ]:
# 첫 글자가 같은 문서 제목을 가져온 뒤, 정규화한 이름으로 찾습니다.
checks = [
    ("Renée Montagne", "R*"),
    ("Émile Driant", "E*"),
]

for requested, pattern in checks:
    candidates = conn.execute(
        "SELECT id FROM documents WHERE id GLOB ?",
        (pattern,)
    ).fetchall()

    wanted = title_key(requested)
    matches = [
        title for (title,) in candidates
        if wanted in title_key(title)
    ]

    print(f"\n요청 제목: {requested!r}")
    print("이름이 포함된 DB 제목:", matches[:20])
    print("가져온 제목 수:", len(candidates))

In [ ]:
import unicodedata
from functools import lru_cache
from blingfire import text_to_sentences

@lru_cache(maxsize=2000)
def get_sentences(title):
    # 악센트 문자의 유니코드 표기 방식이 달라도 찾도록 세 형태를 확인합니다.
    title_forms = list(dict.fromkeys([
        title,
        unicodedata.normalize("NFC", title),
        unicodedata.normalize("NFD", title),
    ]))

    for title_form in title_forms:
        result = conn.execute(
            "SELECT text FROM documents WHERE id = ?",
            (title_form,)
        ).fetchone()

        if result is not None:
            sentences = text_to_sentences(result[0]).splitlines()
            return tuple(s.strip() for s in sentences if s.strip())

    return ()

positive_pairs = []
missing_titles = []
bad_sentence_indices = []

for row in tqdm(train_data):
    for title, sentence_id in row["supporting_facts"]:
        sentences = get_sentences(title)

        if not sentences:
            missing_titles.append(title)
        elif sentence_id >= len(sentences):
            bad_sentence_indices.append(
                (title, sentence_id, len(sentences))
            )
        else:
            positive_pairs.append({
                "claim": row["claim"],
                "title": title,
                "sentence_id": sentence_id,
                "sentence": sentences[sentence_id],
                "relevant": 1,
            })

print("정답 근거 문장 예시 수:", len(positive_pairs))
print("DB에서 문서를 못 찾은 수:", len(missing_titles))
print("문장 번호를 찾지 못한 수:", len(bad_sentence_indices))

In [ ]:
for title, sentence_id, sentence_count in bad_sentence_indices[:5]:
    result = conn.execute(
        "SELECT text FROM documents WHERE id = ?",
        (unicodedata.normalize("NFD", title),)
    ).fetchone()

    # NFD 제목으로 못 찾으면 NFC도 확인합니다.
    if result is None:
        result = conn.execute(
            "SELECT text FROM documents WHERE id = ?",
            (unicodedata.normalize("NFC", title),)
        ).fetchone()

    print(f"\n문서: {title}")
    print(f"HoVer 근거 문장 번호: {sentence_id} / BlingFire 문장 수: {sentence_count}")

    if result is not None:
        print("DB 본문 끝부분:", repr(result[0][-500:]))
        sentences = get_sentences(title)
        start = max(0, len(sentences) - 3)
        print("분리된 마지막 문장들:")
        for i in range(start, len(sentences)):
            print(f"  {i}: {sentences[i]}")

In [ ]:
import shutil
from pathlib import Path

print("Java 경로:", shutil.which("java"))
print("HoVer 폴더의 CoreNLP 파일:")
print(list(Path("/content/hover").glob("*corenlp*")))
print(list(Path("/content/hover").glob("*stanford*")))

!java -version

In [ ]:
%cd /content

!wget -q --show-progress \
  https://nlp.stanford.edu/software/stanford-corenlp-4.6.0.zip \
  -O stanford-corenlp-4.6.0.zip

!unzip -q stanford-corenlp-4.6.0.zip
!pip -q install stanfordcorenlp

from pathlib import Path

corenlp_dir = Path("/content/stanford-corenlp-4.6.0")
print("CoreNLP 폴더:", corenlp_dir.exists())
print("JAR 파일 수:", len(list(corenlp_dir.glob("*.jar"))))

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path("/content/stanford-corenlp-4.6.0.zip")

print("기존 파일 크기:", zip_path.stat().st_size if zip_path.exists() else "파일 없음")
print("파일 앞부분:", zip_path.read_bytes()[:100] if zip_path.exists() else "파일 없음")

!rm -f /content/stanford-corenlp-4.6.0.zip
!curl -fL --retry 3 \
  https://nlp.stanford.edu/software/stanford-corenlp-4.6.0.zip \
  -o /content/stanford-corenlp-4.6.0.zip

zip_path = Path("/content/stanford-corenlp-4.6.0.zip")
print("다운로드 크기:", zip_path.stat().st_size if zip_path.exists() else "파일 없음")
print("ZIP 파일인가:", zipfile.is_zipfile(zip_path))

In [ ]:
!pip -q install -U huggingface_hub

from huggingface_hub import hf_hub_download
from pathlib import Path
import zipfile

zip_path = hf_hub_download(
    repo_id="stanfordnlp/CoreNLP",
    filename="stanford-corenlp-latest.zip",
    local_dir="/content/corenlp-download",
)

print("다운로드 파일 크기:", Path(zip_path).stat().st_size)
print("ZIP 파일인가:", zipfile.is_zipfile(zip_path))

if zipfile.is_zipfile(zip_path):
    !unzip -q "$zip_path" -d /content/corenlp-unpacked
    print("압축 해제 완료")
    print("설치 폴더:", list(Path("/content/corenlp-unpacked").glob("stanford-corenlp-*"))[:3])

In [ ]:
%pip -q install "huggingface_hub>=1.32,<2"

import json
import subprocess
import time
from pathlib import Path

import requests

corenlp_dir = Path("/content/corenlp-unpacked/stanford-corenlp-4.6.0")
log_file = open("/content/corenlp_server.log", "w")

corenlp_process = subprocess.Popen(
    [
        "java", "-mx4g", "-cp", "*",
        "edu.stanford.nlp.pipeline.StanfordCoreNLPServer",
        "-annotators", "tokenize,ssplit",
        "-port", "9000",
        "-timeout", "30000",
    ],
    cwd=str(corenlp_dir),
    stdout=log_file,
    stderr=log_file,
)

test_text = "The first sentence. The second sentence."
properties = {
    "annotators": "tokenize,ssplit",
    "outputFormat": "json",
}

for attempt in range(30):
    if corenlp_process.poll() is not None:
        print("CoreNLP 서버가 종료됐습니다. 로그 마지막 부분:")
        print(Path("/content/corenlp_server.log").read_text()[-3000:])
        break

    try:
        response = requests.post(
            "http://localhost:9000",
            params={"properties": json.dumps(properties)},
            data=test_text.encode("utf-8"),
            headers={"Content-Type": "text/plain; charset=utf-8"},
            timeout=10,
        )
        response.raise_for_status()
        result = response.json()
        print("CoreNLP 연결 성공")
        print("분리된 문장:")
        for sentence in result["sentences"]:
            print("-", sentence["text"])
        break
    except requests.RequestException:
        time.sleep(2)
else:
    print("서버 응답을 받지 못했습니다. 로그 마지막 부분:")
    print(Path("/content/corenlp_server.log").read_text()[-3000:])

In [ ]:
import json
import requests

test_text = "The first sentence. The second sentence."
properties = {
    "annotators": "tokenize,ssplit",
    "outputFormat": "json",
}

try:
    response = requests.post(
        "http://localhost:9000",
        params={"properties": json.dumps(properties)},
        data=test_text.encode("utf-8"),
        headers={"Content-Type": "text/plain; charset=utf-8"},
        timeout=20,
    )
    response.raise_for_status()

    result = response.json()
    print("CoreNLP 서버 연결 성공")
    for sentence in result["sentences"]:
        print("-", sentence["text"])

except Exception as e:
    print("9000번 포트의 서버에 연결하지 못했습니다.")
    print(type(e).__name__, e)

In [ ]:
import json
import subprocess
import time
from pathlib import Path

import requests

corenlp_dir = Path("/content/corenlp-unpacked/stanford-corenlp-4.6.0")
log_path = Path("/content/corenlp_server_9001.log")
log_file = open(log_path, "w")

corenlp_process = subprocess.Popen(
    [
        "java", "-mx4g", "-cp", "*",
        "edu.stanford.nlp.pipeline.StanfordCoreNLPServer",
        "-annotators", "tokenize,ssplit",
        "-port", "9001",
        "-timeout", "30000",
    ],
    cwd=str(corenlp_dir),
    stdout=log_file,
    stderr=log_file,
)

properties = {"annotators": "tokenize,ssplit", "outputFormat": "json"}
test_text = "The first sentence. The second sentence."

for _ in range(30):
    if corenlp_process.poll() is not None:
        print("CoreNLP 서버가 종료됐습니다.")
        print(log_path.read_text()[-3000:])
        break

    try:
        response = requests.post(
            "http://localhost:9001",
            params={"properties": json.dumps(properties)},
            data=test_text.encode("utf-8"),
            headers={"Content-Type": "text/plain; charset=utf-8"},
            timeout=10,
        )
        response.raise_for_status()

        print("CoreNLP 연결 성공")
        for sentence in response.json()["sentences"]:
            print("-", sentence["text"])
        break

    except requests.RequestException:
        time.sleep(2)
else:
    print("서버가 1분 안에 응답하지 않았습니다.")
    print(log_path.read_text()[-3000:])

In [ ]:
result = response.json()

print("응답 항목:", result.keys())

for i, sentence in enumerate(result["sentences"]):
    tokens = sentence.get("tokens", [])
    text = " ".join(
        token.get("originalText", token.get("word", ""))
        for token in tokens
    )

    print(f"\n문장 {i}")
    print("응답 항목:", sentence.keys())
    print(text)

In [ ]:
import json
import unicodedata
from functools import lru_cache

import requests

@lru_cache(maxsize=2000)
def get_sentences_corenlp(title):
    # HoVer와 같이 NFD 제목으로 먼저 DB를 조회합니다.
    title_forms = list(dict.fromkeys([
        unicodedata.normalize("NFD", title),
        unicodedata.normalize("NFC", title),
    ]))

    result = None
    for title_form in title_forms:
        result = conn.execute(
            "SELECT text FROM documents WHERE id = ?",
            (title_form,)
        ).fetchone()
        if result is not None:
            break

    if result is None:
        return ()

    raw_text = result[0]
    properties = {
        "annotators": "tokenize,ssplit",
        "outputFormat": "json",
    }

    response = requests.post(
        "http://localhost:9001",
        params={"properties": json.dumps(properties)},
        data=raw_text.encode("utf-8"),
        headers={"Content-Type": "text/plain; charset=utf-8"},
        timeout=60,
    )
    response.raise_for_status()

    sentences = []
    for sentence in response.json()["sentences"]:
        tokens = sentence["tokens"]
        start = tokens[0]["characterOffsetBegin"]
        end = tokens[-1]["characterOffsetEnd"]
        sentences.append(raw_text[start:end].strip())

    return tuple(sentences)


# 이전에 범위를 벗어났던 문서 5개를 CoreNLP로 다시 확인합니다.
for title, sentence_id, _ in bad_sentence_indices[:5]:
    sentences = get_sentences_corenlp(title)

    print(f"\n문서: {title}")
    print(f"HoVer 근거 번호: {sentence_id} | CoreNLP 문장 수: {len(sentences)}")

    if sentence_id < len(sentences):
        print("해당 번호 문장:", sentences[sentence_id])
    else:
        print("해당 번호는 여전히 범위 밖입니다.")

In [ ]:
checks = [
    ("Life Goes On (Fergie song)", 2),
    ("Fas'h mosque", 4),
    ("Terry Ray (actor)", 7),
    ("Panther tank", 4),
]

for title, sentence_id in checks:
    sentences = get_sentences_corenlp(title)

    print(f"\n문서: {title}")
    print(f"HoVer 근거 번호: {sentence_id} | CoreNLP 문장 수: {len(sentences)}")

    for i, sentence in enumerate(sentences):
        marker = " ← 근거 번호" if i == sentence_id else ""
        print(f"{i}: {sentence}{marker}")

In [ ]:
# 학습 데이터의 supporting_facts에 나오는 고유 문서 목록
gold_titles = sorted({
    title
    for row in train_data
    for title, _ in row["supporting_facts"]
})

print("고유 정답 문서 수:", len(gold_titles))

# 문서마다 CoreNLP 문장 분리를 한 번씩 실행
sentences_by_title = {}
corenlp_errors = []

for title in tqdm(gold_titles):
    try:
        sentences_by_title[title] = get_sentences_corenlp(title)
    except Exception as e:
        corenlp_errors.append((title, str(e)))

# supporting_facts의 문장 번호로 정답 문장 예시 구성
positive_pairs = []
missing_titles = []
bad_sentence_indices = []

for row in train_data:
    for title, sentence_id in row["supporting_facts"]:
        sentences = sentences_by_title.get(title, ())

        if not sentences:
            missing_titles.append(title)
        elif sentence_id < 0 or sentence_id >= len(sentences):
            bad_sentence_indices.append(
                (title, sentence_id, len(sentences))
            )
        else:
            positive_pairs.append({
                "claim": row["claim"],
                "title": title,
                "sentence_id": sentence_id,
                "sentence": sentences[sentence_id],
                "relevant": 1,
            })

print("\n정답 근거 문장 예시 수:", len(positive_pairs))
print("문서를 찾지 못한 수:", len(missing_titles))
print("문장 번호가 범위를 벗어난 수:", len(bad_sentence_indices))
print("CoreNLP 처리 오류 문서 수:", len(corenlp_errors))

if positive_pairs:
    print("\n예시:")
    print("Claim:", positive_pairs[0]["claim"])
    print("문장:", positive_pairs[0]["sentence"])

In [ ]:
sample = train_data[0]
retrieval_sample = retrieval_by_claim[sample["claim"]]
doc_results = retrieval_sample["doc_retrieval_results"]

print("검색 결과 바깥쪽 자료형:", type(doc_results).__name__)
print("검색 결과 바깥쪽 길이:", len(doc_results))
print("첫 항목 자료형:", type(doc_results[0]).__name__)
print("첫 항목 길이:", len(doc_results[0]))

print("\n상위 10개 문서 제목:")
for title in doc_results[0][0][:10]:
    print("-", title)

print("\n상위 10개 점수:", doc_results[0][1][:10])

In [ ]:
import unicodedata

def normalize_title(title):
    title = unicodedata.normalize("NFKD", title)
    title = "".join(
        char for char in title
        if not unicodedata.combining(char)
    )
    return title.casefold().strip()

for k in [3, 5, 10, 20, 100]:
    claims_with_any = 0
    claims_with_all = 0
    gold_docs_found = 0
    gold_docs_total = 0

    for row in train_data:
        retrieval = retrieval_by_claim[row["claim"]]
        retrieved_titles = retrieval["doc_retrieval_results"][0][0][:k]

        retrieved_set = {normalize_title(t) for t in retrieved_titles}
        gold_set = {
            normalize_title(title)
            for title, _ in row["supporting_facts"]
        }

        found = retrieved_set & gold_set
        claims_with_any += bool(found)
        claims_with_all += found == gold_set
        gold_docs_found += len(found)
        gold_docs_total += len(gold_set)

    print(
        f"상위 {k:>3}개 문서 | "
        f"정답 문서 하나 이상 포함: {claims_with_any / len(train_data):.1%} | "
        f"정답 문서 전부 포함: {claims_with_all / len(train_data):.1%} | "
        f"정답 문서 재현율: {gold_docs_found / gold_docs_total:.1%}"
    )

In [ ]:
for k in [3, 10, 20, 100]:
    unique_titles = set()

    for row in train_data:
        retrieval = retrieval_by_claim[row["claim"]]
        titles = retrieval["doc_retrieval_results"][0][0][:k]
        unique_titles.update(titles)

    print(
        f"상위 {k:>3}개 기준 | "
        f"고유 문서 수: {len(unique_titles):,} | "
        f"claim-문서 후보 수: {len(train_data) * k:,}"
    )

In [ ]:
TOP_K = 20

candidate_titles = sorted({
    title
    for row in train_data
    for title in retrieval_by_claim[row["claim"]]["doc_retrieval_results"][0][0][:TOP_K]
})

print("처리할 고유 후보 문서 수:", len(candidate_titles))

candidate_sentences = {}
candidate_errors = []

for title in tqdm(candidate_titles):
    try:
        candidate_sentences[title] = get_sentences_corenlp(title)
    except Exception as e:
        candidate_errors.append((title, str(e)))

print("\n문장 분리 완료 문서 수:", len(candidate_sentences))
print("처리 오류 문서 수:", len(candidate_errors))

if candidate_errors:
    print("오류 예시:", candidate_errors[:3])

In [ ]:
import html
import re
import unicodedata
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

def normalize_title(title):
    title = unicodedata.normalize("NFKD", title)
    title = "".join(
        char for char in title
        if not unicodedata.combining(char)
    )
    return title.casefold().strip()

def clean_sentence(text):
    # 문장 번호를 찾은 뒤 모델 입력에서만 HTML 태그를 정리합니다.
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", html.unescape(text)).strip()

NEGATIVES_PER_CLAIM = 5
pair_examples = []
claims_with_positive = 0
claims_without_positive = 0

for row in tqdm(train_data):
    claim = row["claim"]
    retrieved_titles = retrieval_by_claim[claim]["doc_retrieval_results"][0][0][:TOP_K]

    gold_facts = {
        (normalize_title(title), sentence_id)
        for title, sentence_id in row["supporting_facts"]
    }
    gold_doc_titles = {title for title, _ in gold_facts}

    positives = []
    negative_candidates = []

    for title in retrieved_titles:
        title_key = normalize_title(title)
        sentences = candidate_sentences.get(title, ())

        for sentence_id, sentence in enumerate(sentences):
            text = clean_sentence(sentence)
            if not text:
                continue

            item = {
                "claim": claim,
                "title": title,
                "sentence_id": sentence_id,
                "sentence": text,
            }

            if (title_key, sentence_id) in gold_facts:
                positives.append(item)
            elif title_key not in gold_doc_titles:
                # 정답 문서가 아닌 문서에서만 negative 후보를 만듭니다.
                negative_candidates.append(item)

    if not positives:
        claims_without_positive += 1
        continue

    claims_with_positive += 1

    for item in positives:
        pair_examples.append({**item, "relevant": 1})

    if negative_candidates:
        texts = [claim] + [item["sentence"] for item in negative_candidates]
        matrix = TfidfVectorizer(
            ngram_range=(1, 2),
            stop_words="english"
        ).fit_transform(texts)

        scores = cosine_similarity(matrix[0:1], matrix[1:]).ravel()
        top_indices = np.argsort(scores)[::-1][:NEGATIVES_PER_CLAIM]

        for index in top_indices:
            pair_examples.append({
                **negative_candidates[index],
                "relevant": 0,
            })

positive_count = sum(x["relevant"] == 1 for x in pair_examples)
negative_count = sum(x["relevant"] == 0 for x in pair_examples)

print("학습에 positive가 있는 claim:", claims_with_positive)
print("후보 안에 positive가 없는 claim:", claims_without_positive)
print("positive 문장 쌍:", positive_count)
print("negative 문장 쌍:", negative_count)
print("전체 학습 쌍:", len(pair_examples))

In [ ]:
VAL_TOP_K = TOP_K

val_candidate_titles = sorted({
    title
    for row in val_data
    for title in retrieval_by_claim[row["claim"]]["doc_retrieval_results"][0][0][:VAL_TOP_K]
})

print("검증용 고유 후보 문서 수:", len(val_candidate_titles))

val_candidate_sentences = {}
val_corenlp_errors = []

for title in tqdm(val_candidate_titles):
    try:
        val_candidate_sentences[title] = get_sentences_corenlp(title)
    except Exception as e:
        val_corenlp_errors.append((title, str(e)))

print("\n문장 분리 완료 문서 수:", len(val_candidate_sentences))
print("CoreNLP 처리 오류 문서 수:", len(val_corenlp_errors))

# 각 검증 claim의 후보 문장 전체에 정답 여부를 붙입니다.
val_candidates_by_uid = {}

for row in tqdm(val_data):
    claim = row["claim"]
    gold_facts = {
        (normalize_title(title), sentence_id)
        for title, sentence_id in row["supporting_facts"]
    }

    candidates = []
    titles = retrieval_by_claim[claim]["doc_retrieval_results"][0][0][:VAL_TOP_K]

    for title in titles:
        title_key = normalize_title(title)

        for sentence_id, sentence in enumerate(
            val_candidate_sentences.get(title, ())
        ):
            text = clean_sentence(sentence)
            if text:
                candidates.append({
                    "claim": claim,
                    "title": title,
                    "sentence_id": sentence_id,
                    "sentence": text,
                    "relevant": int((title_key, sentence_id) in gold_facts),
                })

    val_candidates_by_uid[row["uid"]] = candidates

print("\n검증 claim 수:", len(val_candidates_by_uid))
print(
    "후보 안에 정답 문장이 하나 이상 있는 claim:",
    sum(
        any(item["relevant"] for item in candidates)
        for candidates in val_candidates_by_uid.values()
    )
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm
import numpy as np

# 학습 쌍: Claim과 Evidence 문장을 이어 붙여 관련도를 학습
train_texts = [
    f"Claim: {x['claim']} Evidence: {x['sentence']}"
    for x in pair_examples
]
y_train = np.array([x["relevant"] for x in pair_examples])

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=120_000,
)
X_train = vectorizer.fit_transform(train_texts)

ranker = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    solver="liblinear",
)
ranker.fit(X_train, y_train)

print("학습 완료")
print("학습 쌍:", len(y_train))
print("사용한 단어 특징 수:", X_train.shape[1])

# 검증 후보 문장 전체의 관련도 점수 계산
val_flat = [
    (uid, candidate)
    for uid, candidates in val_candidates_by_uid.items()
    for candidate in candidates
]

val_texts = [
    f"Claim: {candidate['claim']} Evidence: {candidate['sentence']}"
    for _, candidate in val_flat
]
X_val = vectorizer.transform(val_texts)
val_scores = ranker.predict_proba(X_val)[:, 1]

# claim별로 점수 순으로 정렬
ranked_by_uid = {}
offset = 0

for uid, candidates in val_candidates_by_uid.items():
    end = offset + len(candidates)
    scores = val_scores[offset:end]

    ranked_by_uid[uid] = [
        candidates[i]
        for i in np.argsort(scores)[::-1]
    ]
    offset = end

# HoVer 정답 근거와 비교해 순위 평가
gold_by_uid = {
    row["uid"]: {
        (normalize_title(title), sentence_id)
        for title, sentence_id in row["supporting_facts"]
    }
    for row in val_data
}

gold_total = sum(len(facts) for facts in gold_by_uid.values())
candidate_gold_total = 0
candidate_any = 0
candidate_all = 0

for uid, candidates in val_candidates_by_uid.items():
    candidate_keys = {
        (normalize_title(x["title"]), x["sentence_id"])
        for x in candidates
    }
    gold = gold_by_uid[uid]
    found = candidate_keys & gold

    candidate_gold_total += len(found)
    candidate_any += bool(found)
    candidate_all += (gold <= candidate_keys)

print(f"\n후보 문서에서 정답 문장을 찾을 수 있는 claim: {candidate_any}/{len(val_data)}")
print(f"후보 안에 정답 근거가 전부 있는 claim: {candidate_all}/{len(val_data)}")
print(f"후보 문장 Recall 상한: {candidate_gold_total / gold_total:.3f}")

for k in [1, 3, 5, 10]:
    fact_hits = 0
    claims_with_all = 0
    reciprocal_ranks = []

    for uid, ranked in ranked_by_uid.items():
        gold = gold_by_uid[uid]
        top_keys = {
            (normalize_title(x["title"]), x["sentence_id"])
            for x in ranked[:k]
        }

        fact_hits += len(gold & top_keys)
        claims_with_all += (gold <= top_keys)

        first_rank = next(
            (
                rank
                for rank, item in enumerate(ranked, start=1)
                if (normalize_title(item["title"]), item["sentence_id"]) in gold
            ),
            None,
        )
        reciprocal_ranks.append(
            1 / first_rank if first_rank is not None else 0
        )

    print(
        f"Top-{k:>2} | "
        f"근거 문장 Recall: {fact_hits / gold_total:.3f} | "
        f"정답 근거 전부 포함한 claim: {claims_with_all / len(val_data):.3f} | "
        f"MRR: {np.mean(reciprocal_ranks):.3f}"
    )

In [ ]:
# 학습 쌍의 claim과 sentence로 TF-IDF 공간을 만듭니다.
similarity_texts = []
for item in pair_examples:
    similarity_texts.append(item["claim"])
    similarity_texts.append(item["sentence"])

similarity_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=120_000,
)
similarity_vectorizer.fit(similarity_texts)

# 각 claim과 후보 문장의 코사인 유사도 계산
val_claim_matrix = similarity_vectorizer.transform(
    [candidate["claim"] for _, candidate in val_flat]
)
val_sentence_matrix = similarity_vectorizer.transform(
    [candidate["sentence"] for _, candidate in val_flat]
)
tfidf_scores = np.asarray(
    val_claim_matrix.multiply(val_sentence_matrix).sum(axis=1)
).ravel()


def evaluate_ranking(scores, method_name):
    ranked_by_uid = {}
    offset = 0

    for uid, candidates in val_candidates_by_uid.items():
        end = offset + len(candidates)
        order = np.argsort(scores[offset:end])[::-1]
        ranked_by_uid[uid] = [candidates[i] for i in order]
        offset = end

    print(f"\n[{method_name}]")

    for k in [1, 3, 5, 10]:
        fact_hits = 0
        claims_with_all = 0

        for uid, ranked in ranked_by_uid.items():
            gold = gold_by_uid[uid]
            top_keys = {
                (normalize_title(x["title"]), x["sentence_id"])
                for x in ranked[:k]
            }

            fact_hits += len(gold & top_keys)
            claims_with_all += (gold <= top_keys)

        print(
            f"Top-{k:>2} | "
            f"근거 문장 Recall: {fact_hits / gold_total:.3f} | "
            f"정답 근거 전부 포함한 claim: "
            f"{claims_with_all / len(val_data):.3f}"
        )


evaluate_ranking(tfidf_scores, "단순 TF-IDF 코사인 유사도")

In [ ]:
# 같은 TF-IDF 공간에서 claim과 문장을 각각 벡터화한 뒤,
# 겹치는 단어 특징만 남깁니다.
train_claim_matrix = similarity_vectorizer.transform(
    [x["claim"] for x in pair_examples]
)
train_sentence_matrix = similarity_vectorizer.transform(
    [x["sentence"] for x in pair_examples]
)
X_train_overlap = train_claim_matrix.multiply(train_sentence_matrix)

y_train = np.array([x["relevant"] for x in pair_examples])

overlap_ranker = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    solver="liblinear",
)
overlap_ranker.fit(X_train_overlap, y_train)

# 검증 후보도 같은 방식으로 변환하고 점수를 계산합니다.
val_claim_matrix = similarity_vectorizer.transform(
    [candidate["claim"] for _, candidate in val_flat]
)
val_sentence_matrix = similarity_vectorizer.transform(
    [candidate["sentence"] for _, candidate in val_flat]
)
X_val_overlap = val_claim_matrix.multiply(val_sentence_matrix)

overlap_scores = overlap_ranker.predict_proba(X_val_overlap)[:, 1]

evaluate_ranking(overlap_scores, "단어 겹침 특징을 학습한 Logistic Regression")

In [ ]:
import torch

print("GPU 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("현재 런타임은 CPU입니다.")

In [ ]:
%pip -q install sentence-transformers "huggingface_hub>=1.32,<2"

In [ ]:
import torch
from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder, InputExample

# 준비한 학습 쌍의 형태 확인
print("학습 쌍 수:", len(pair_examples))
print("예시:", pair_examples[0])

# claim과 문장, 관련성 라벨(0 또는 1)을 모델 학습 형식으로 변환
train_examples = [
    InputExample(
        texts=[item["claim"], item["sentence"]],
        label=float(item["relevant"])
    )
    for item in pair_examples
]

train_loader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=32
)

# 문장 재순위 사전학습 모델을 HoVer 학습 쌍으로 미세조정
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    num_labels=1,
    max_length=256,
    device="cuda"
)

cross_encoder.fit(
    train_dataloader=train_loader,
    epochs=1,
    warmup_steps=int(len(train_loader) * 0.1),
    optimizer_params={"lr": 2e-5},
    output_path="/content/hover_cross_encoder_reranker",
    use_amp=True,
    show_progress_bar=True
)

print("학습 완료")

In [ ]:
import numpy as np
from tqdm.auto import tqdm

# 각 검증 후보를 (claim, 문장) 쌍으로 만들기
val_pairs = [
    (candidate["claim"], candidate["sentence"])
    for _, candidate in val_flat
]

# GPU 메모리 부담을 줄이기 위해 나눠서 점수 계산
cross_scores = []

batch_size = 128
chunk_size = 10000

for start in tqdm(range(0, len(val_pairs), chunk_size)):
    batch_pairs = val_pairs[start:start + chunk_size]

    batch_scores = cross_encoder.predict(
        batch_pairs,
        batch_size=batch_size,
        show_progress_bar=False
    )

    cross_scores.extend(np.asarray(batch_scores).reshape(-1))

cross_scores = np.asarray(cross_scores)

print("채점한 검증 문장 수:", len(cross_scores))
print("점수 예시:", cross_scores[:5])

# 기존 평가 함수로 TF-IDF와 같은 기준에서 평가
evaluate_ranking(cross_scores, "HoVer 미세조정 Cross-Encoder")

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import CrossEncoder

print("평가 모델: 미세조정 전 사전학습 Cross-Encoder")

base_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    num_labels=1,
    max_length=256,
    device="cuda"
)

base_scores = []

for start in tqdm(range(0, len(val_pairs), 10000)):
    batch_pairs = val_pairs[start:start + 10000]
    batch_scores = base_encoder.predict(
        batch_pairs,
        batch_size=128,
        show_progress_bar=False
    )
    base_scores.extend(np.asarray(batch_scores).reshape(-1))

base_scores = np.asarray(base_scores)

print("채점한 문장 수:", len(base_scores))
print("사전학습 모델 점수 예시:", base_scores[:5])
print("미세조정 모델 점수 예시:", cross_scores[:5])
print("두 모델 점수 평균 차이:", np.mean(np.abs(base_scores - cross_scores)))

evaluate_ranking(base_scores, "미세조정 전 사전학습 Cross-Encoder")

In [ ]:
from collections import defaultdict

# Cross-Encoder 점수와 검증 후보 연결
ranked_by_uid = defaultdict(list)

for (uid, candidate), score in zip(val_flat, base_scores):
    ranked_by_uid[uid].append({
        "title": candidate["title"],
        "sentence_id": candidate["sentence_id"],
        "sentence": candidate["sentence"],
        "score": float(score),
    })

# claim마다 점수가 높은 문장 순서로 정렬
TOP_K_EVIDENCE = 10
val_verifier_inputs = {}

for uid, candidates in ranked_by_uid.items():
    ranked = sorted(
        candidates,
        key=lambda x: x["score"],
        reverse=True
    )[:TOP_K_EVIDENCE]

    # 최종 판정 모델에 넣을 텍스트
    evidence_text = "\n".join(
        f"Evidence {i+1} [{item['title']}, 문장 {item['sentence_id']}]: "
        f"{item['sentence']}"
        for i, item in enumerate(ranked)
    )

    # claim은 후보 중 하나에서 가져옴
    claim = next(
        candidate["claim"]
        for uid2, candidate in val_flat
        if uid2 == uid
    )

    val_verifier_inputs[uid] = {
        "claim": claim,
        "evidence_text": evidence_text,
        "ranked_evidence": ranked,
    }

print("최종 판정 입력 수:", len(val_verifier_inputs))

# 예시 확인
example_uid = next(iter(val_verifier_inputs))
example = val_verifier_inputs[example_uid]

print("\nClaim:")
print(example["claim"])

print("\n최종 판정에 사용할 근거:")
print(example["evidence_text"])

In [ ]:
data_candidates = {}

for name, value in list(globals().items()):
    if (
        isinstance(value, list)
        and len(value) > 0
        and isinstance(value[0], dict)
    ):
        keys = set(value[0].keys())

        if {"uid", "label"}.issubset(keys):
            hop_count = sum(
                row.get("num_hops") == 2
                for row in value
            )

            data_candidates[name] = {
                "전체 수": len(value),
                "2-hop 수": hop_count,
                "필드": list(value[0].keys())
            }

print("후보 데이터 변수:")
for name, info in data_candidates.items():
    print(name, "→", info)

In [ ]:
label_by_uid = {
    row["uid"]: row["label"]
    for row in val_data
}

for uid, item in val_verifier_inputs.items():
    item["label"] = label_by_uid[uid]

print("라벨이 붙은 검증 입력 수:", len(val_verifier_inputs))

example_uid = next(iter(val_verifier_inputs))
print("첫 샘플 라벨:", val_verifier_inputs[example_uid]["label"])

In [ ]:
from collections import defaultdict
import numpy as np
from tqdm.auto import tqdm

# 학습 문장 쌍을 Cross-Encoder 입력 형태로 변환
train_pairs_for_verifier = [
    (item["claim"], item["sentence"])
    for item in pair_examples
]

train_pair_scores = []

for start in tqdm(
    range(0, len(train_pairs_for_verifier), 10000),
    desc="학습 문장 점수 계산"
):
    batch_pairs = train_pairs_for_verifier[start:start + 10000]

    scores = base_encoder.predict(
        batch_pairs,
        batch_size=128,
        show_progress_bar=False
    )

    train_pair_scores.extend(
        np.asarray(scores).reshape(-1)
    )

train_pair_scores = np.asarray(train_pair_scores)

print("점수 계산 완료:", len(train_pair_scores))

In [ ]:
train_ranked_by_claim = defaultdict(list)

for item, score in zip(pair_examples, train_pair_scores):
    train_ranked_by_claim[item["claim"]].append({
        "title": item["title"],
        "sentence_id": item["sentence_id"],
        "sentence": item["sentence"],
        "score": float(score),
    })

train_label_by_claim = {
    row["claim"]: row["label"]
    for row in train_data
}

train_verifier_inputs = {}

for claim, candidates in train_ranked_by_claim.items():
    ranked = sorted(
        candidates,
        key=lambda x: x["score"],
        reverse=True
    )[:10]

    evidence_text = "\n".join(
        f"Evidence {i+1} [{item['title']}, 문장 {item['sentence_id']}]: "
        f"{item['sentence']}"
        for i, item in enumerate(ranked)
    )

    train_verifier_inputs[claim] = {
        "claim": claim,
        "evidence_text": evidence_text,
        "ranked_evidence": ranked,
        "label": train_label_by_claim[claim],
    }

print("최종 판정 학습 입력 수:", len(train_verifier_inputs))

In [ ]:
print("학습 라벨 분포:")
from collections import Counter

print(
    Counter(
        item["label"]
        for item in train_verifier_inputs.values()
    )
)

print("\n학습 샘플:")
train_example = next(iter(train_verifier_inputs.values()))

print("Claim:")
print(train_example["claim"])

print("\nEvidence:")
print(train_example["evidence_text"])

print("\nLabel:")
print(train_example["label"])

In [ ]:
all_train_claims = {
    row["claim"]
    for row in train_data
}

built_train_claims = set(train_verifier_inputs.keys())

missing_claims = all_train_claims - built_train_claims

print("전체 학습 claim 수:", len(all_train_claims))
print("판정 입력이 생성된 claim 수:", len(built_train_claims))
print("누락된 claim 수:", len(missing_claims))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

# 학습 문장 구성
train_texts = [
    f"Claim: {item['claim']}\n{item['evidence_text']}"
    for item in train_verifier_inputs.values()
]

train_labels = [
    item["label"]
    for item in train_verifier_inputs.values()
]

# 검증 문장 구성
val_texts = [
    f"Claim: {item['claim']}\n{item['evidence_text']}"
    for item in val_verifier_inputs.values()
]

val_labels = [
    item["label"]
    for item in val_verifier_inputs.values()
]

# TF-IDF 벡터화
verifier_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=120000,
    sublinear_tf=True
)

X_train = verifier_vectorizer.fit_transform(train_texts)
X_val = verifier_vectorizer.transform(val_texts)

print("학습 행렬:", X_train.shape)
print("검증 행렬:", X_val.shape)

# 최종 판정 모델
verifier_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

verifier_model.fit(X_train, train_labels)

# 검증
val_predictions = verifier_model.predict(X_val)

print("\nAccuracy:", accuracy_score(val_labels, val_predictions))
print(
    "Macro-F1:",
    f1_score(val_labels, val_predictions, average="macro")
)

print("\n라벨별 결과:")
print(
    classification_report(
        val_labels,
        val_predictions,
        labels=["SUPPORTED", "NOT_SUPPORTED"]
    )
)

print("\n혼동 행렬:")
print(
    confusion_matrix(
        val_labels,
        val_predictions,
        labels=["SUPPORTED", "NOT_SUPPORTED"]
    )
)

In [ ]:
import random
import numpy as np
import torch

from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder, InputExample
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 학습 입력을 리스트로 변환
train_items = list(train_verifier_inputs.values())

supported_items = [
    item for item in train_items
    if item["label"] == "SUPPORTED"
]

not_supported_items = [
    item for item in train_items
    if item["label"] == "NOT_SUPPORTED"
]

print("기존 SUPPORTED:", len(supported_items))
print("기존 NOT_SUPPORTED:", len(not_supported_items))

# NOT_SUPPORTED를 oversampling하여 학습 균형 조정
not_supported_balanced = resample(
    not_supported_items,
    replace=True,
    n_samples=len(supported_items),
    random_state=42
)

balanced_train_items = supported_items + not_supported_balanced
random.Random(42).shuffle(balanced_train_items)

# Claim + Evidence 전체를 하나의 입력 쌍으로 구성
verifier_train_examples = [
    InputExample(
        texts=[
            item["claim"],
            item["evidence_text"]
        ],
        label=1.0 if item["label"] == "SUPPORTED" else 0.0
    )
    for item in balanced_train_items
]

verifier_loader = DataLoader(
    verifier_train_examples,
    shuffle=True,
    batch_size=16
)

print("균형 조정 후 학습 쌍:", len(verifier_train_examples))

In [ ]:
# 근거 재순위 모델과 별도의 새 모델
verifier_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    num_labels=1,
    max_length=512,
    device="cuda"
)

verifier_encoder.fit(
    train_dataloader=verifier_loader,
    epochs=1,
    warmup_steps=int(len(verifier_loader) * 0.1),
    optimizer_params={"lr": 2e-5},
    output_path="/content/hover_final_verifier",
    use_amp=True,
    show_progress_bar=True
)

print("최종 판정 모델 학습 완료")

In [ ]:
# 검증 입력 구성
val_items = list(val_verifier_inputs.values())

val_pairs_for_verifier = [
    (item["claim"], item["evidence_text"])
    for item in val_items
]

val_true_labels = [
    1 if item["label"] == "SUPPORTED" else 0
    for item in val_items
]

# 예측 점수 계산
val_scores = verifier_encoder.predict(
    val_pairs_for_verifier,
    batch_size=32,
    show_progress_bar=True
)

val_scores = np.asarray(val_scores).reshape(-1)

print("점수 범위:", val_scores.min(), "~", val_scores.max())

# 모델 출력이 logit인지 확률인지 자동 처리
if val_scores.min() < 0 or val_scores.max() > 1:
    val_probabilities = 1 / (1 + np.exp(-val_scores))
else:
    val_probabilities = val_scores

val_pred_labels = (val_probabilities >= 0.5).astype(int)

print("\nAccuracy:", accuracy_score(val_true_labels, val_pred_labels))
print(
    "Macro-F1:",
    f1_score(val_true_labels, val_pred_labels, average="macro")
)

print("\n라벨별 결과:")
print(
    classification_report(
        val_true_labels,
        val_pred_labels,
        target_names=["NOT_SUPPORTED", "SUPPORTED"]
    )
)

print("\n혼동 행렬:")
print(
    confusion_matrix(
        val_true_labels,
        val_pred_labels
    )
)

In [ ]:
from collections import defaultdict
import numpy as np
from tqdm.auto import tqdm

print("retrieval_by_claim 존재:", "retrieval_by_claim" in globals())
print("candidate_sentences 존재:", "candidate_sentences" in globals())

# 전체 학습 claim과 현재 입력이 만들어진 claim
all_train_claims = {
    row["claim"]
    for row in train_data
}

built_claims = set(train_verifier_inputs.keys())
missing_claims = all_train_claims - built_claims

print("누락 claim 수:", len(missing_claims))

# claim → 라벨
train_label_by_claim = {
    row["claim"]: row["label"]
    for row in train_data
}

# 후보 문장 검색
def get_candidate_sentences(title):
    if title in candidate_sentences:
        return candidate_sentences[title]

    # 혹시 제목의 악센트 표기가 다른 경우를 대비
    title_norm = normalize_title(title)

    for saved_title, sentences in candidate_sentences.items():
        if normalize_title(saved_title) == title_norm:
            return sentences

    return []

missing_candidates = {}

for claim in tqdm(missing_claims, desc="누락 claim 후보 생성"):
    retrieval_row = retrieval_by_claim.get(claim)

    if retrieval_row is None:
        continue

    retrieval_result = retrieval_row["doc_retrieval_results"]

    # 구조: [[문서 제목 목록, 점수 목록], 0]
    titles = retrieval_result[0][0][:20]

    candidates = []

    for title in titles:
        sentences = get_candidate_sentences(title)

        for sentence_id, sentence in enumerate(sentences):
            candidates.append({
                "title": title,
                "sentence_id": sentence_id,
                "sentence": sentence
            })

    if candidates:
        missing_candidates[claim] = candidates

print("후보 문장이 생성된 누락 claim:", len(missing_candidates))

In [ ]:
missing_pairs = [
    (claim, candidate["sentence"])
    for claim, candidates in missing_candidates.items()
    for candidate in candidates
]

missing_scores = []

for start in tqdm(
    range(0, len(missing_pairs), 10000),
    desc="누락 문장 점수 계산"
):
    batch_pairs = missing_pairs[start:start + 10000]

    scores = base_encoder.predict(
        batch_pairs,
        batch_size=128,
        show_progress_bar=False
    )

    missing_scores.extend(
        np.asarray(scores).reshape(-1)
    )

missing_scores = np.asarray(missing_scores)

print("점수 계산 문장 수:", len(missing_scores))

In [ ]:
score_index = 0
added_count = 0

for claim, candidates in missing_candidates.items():
    scored_candidates = []

    for candidate in candidates:
        candidate_with_score = dict(candidate)
        candidate_with_score["score"] = float(
            missing_scores[score_index]
        )

        scored_candidates.append(candidate_with_score)
        score_index += 1

    ranked = sorted(
        scored_candidates,
        key=lambda x: x["score"],
        reverse=True
    )[:10]

    evidence_text = "\n".join(
        f"Evidence {i+1} [{item['title']}, 문장 {item['sentence_id']}]: "
        f"{item['sentence']}"
        for i, item in enumerate(ranked)
    )

    train_verifier_inputs[claim] = {
        "claim": claim,
        "evidence_text": evidence_text,
        "ranked_evidence": ranked,
        "label": train_label_by_claim[claim],
    }

    added_count += 1

print("추가된 학습 입력:", added_count)
print("최종 학습 입력 수:", len(train_verifier_inputs))

In [ ]:
all_train_claims = {
    row["claim"]
    for row in train_data
}

remaining_claims = all_train_claims - set(
    train_verifier_inputs.keys()
)

print("아직 누락된 claim 수:", len(remaining_claims))

for claim in list(remaining_claims)[:5]:
    print("-", claim)

In [ ]:
import random
from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder, InputExample
from sklearn.utils import resample

# 최신 학습 입력
train_items = list(train_verifier_inputs.values())

supported_items = [
    item for item in train_items
    if item["label"] == "SUPPORTED"
]

not_supported_items = [
    item for item in train_items
    if item["label"] == "NOT_SUPPORTED"
]

print("SUPPORTED:", len(supported_items))
print("NOT_SUPPORTED:", len(not_supported_items))

# 라벨 균형 조정
not_supported_balanced = resample(
    not_supported_items,
    replace=True,
    n_samples=len(supported_items),
    random_state=42
)

balanced_train_items = supported_items + not_supported_balanced
random.Random(42).shuffle(balanced_train_items)

verifier_train_examples = [
    InputExample(
        texts=[
            item["claim"],
            item["evidence_text"]
        ],
        label=1.0 if item["label"] == "SUPPORTED" else 0.0
    )
    for item in balanced_train_items
]

verifier_loader = DataLoader(
    verifier_train_examples,
    shuffle=True,
    batch_size=16
)

print("균형 조정 후 학습 쌍:", len(verifier_train_examples))

In [ ]:
final_verifier = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    num_labels=1,
    max_length=512,
    device="cuda"
)

final_verifier.fit(
    train_dataloader=verifier_loader,
    epochs=1,
    warmup_steps=int(len(verifier_loader) * 0.1),
    optimizer_params={"lr": 2e-5},
    output_path="/content/hover_final_verifier_full",
    use_amp=True,
    show_progress_bar=True
)

print("전체 학습 입력 기반 최종 판정 모델 학습 완료")

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

val_items = list(val_verifier_inputs.values())

val_pairs = [
    (item["claim"], item["evidence_text"])
    for item in val_items
]

val_true = np.array([
    1 if item["label"] == "SUPPORTED" else 0
    for item in val_items
])

val_scores = final_verifier.predict(
    val_pairs,
    batch_size=32,
    show_progress_bar=True
)

val_scores = np.asarray(val_scores).reshape(-1)

print("점수 범위:", val_scores.min(), "~", val_scores.max())

# 점수가 logit이면 확률로 변환
if val_scores.min() < 0 or val_scores.max() > 1:
    val_probabilities = 1 / (1 + np.exp(-val_scores))
else:
    val_probabilities = val_scores

val_pred = (val_probabilities >= 0.5).astype(int)

print("\nAccuracy:", accuracy_score(val_true, val_pred))
print(
    "Macro-F1:",
    f1_score(val_true, val_pred, average="macro")
)

print("\n라벨별 결과:")
print(
    classification_report(
        val_true,
        val_pred,
        target_names=["NOT_SUPPORTED", "SUPPORTED"]
    )
)

print("\n혼동 행렬:")
print(
    confusion_matrix(
        val_true,
        val_pred
    )
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

threshold_results = []

for threshold in np.arange(0.30, 0.71, 0.05):
    predictions = (
        val_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": round(float(threshold), 2),
        "accuracy": accuracy_score(val_true, predictions),
        "macro_f1": f1_score(
            val_true,
            predictions,
            average="macro"
        )
    })

for result in threshold_results:
    print(
        f"threshold={result['threshold']:.2f} | "
        f"accuracy={result['accuracy']:.4f} | "
        f"macro_f1={result['macro_f1']:.4f}"
    )

In [ ]:
import gc
import random
import numpy as np
import torch

from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder, InputExample
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, f1_score

def build_evidence_text(item, top_k):
    selected = item["ranked_evidence"][:top_k]

    return "\n".join(
        f"Evidence {i+1} [{e['title']}, 문장 {e['sentence_id']}]: "
        f"{e['sentence']}"
        for i, e in enumerate(selected)
    )

# 학습 데이터 균형 조정
train_items = list(train_verifier_inputs.values())

supported_items = [
    item for item in train_items
    if item["label"] == "SUPPORTED"
]

not_supported_items = [
    item for item in train_items
    if item["label"] == "NOT_SUPPORTED"
]

not_supported_balanced = resample(
    not_supported_items,
    replace=True,
    n_samples=len(supported_items),
    random_state=42
)

balanced_train_items = supported_items + not_supported_balanced
random.Random(42).shuffle(balanced_train_items)

results = []

for top_k in [3, 5, 10]:

    print(f"\n===== Top-{top_k} 학습 시작 =====")

    train_examples = [
        InputExample(
            texts=[
                item["claim"],
                build_evidence_text(item, top_k)
            ],
            label=1.0 if item["label"] == "SUPPORTED" else 0.0
        )
        for item in balanced_train_items
    ]

    train_loader = DataLoader(
        train_examples,
        shuffle=True,
        batch_size=16
    )

    model = CrossEncoder(
        "cross-encoder/ms-marco-MiniLM-L6-v2",
        num_labels=1,
        max_length=512,
        device="cuda"
    )

    model.fit(
        train_dataloader=train_loader,
        epochs=1,
        warmup_steps=int(len(train_loader) * 0.1),
        optimizer_params={"lr": 2e-5},
        output_path=f"/content/hover_final_verifier_top{top_k}",
        use_amp=True,
        show_progress_bar=True
    )

    # 검증
    val_items = list(val_verifier_inputs.values())

    val_pairs = [
        (
            item["claim"],
            build_evidence_text(item, top_k)
        )
        for item in val_items
    ]

    val_true = np.array([
        1 if item["label"] == "SUPPORTED" else 0
        for item in val_items
    ])

    scores = model.predict(
        val_pairs,
        batch_size=32,
        show_progress_bar=True
    )

    scores = np.asarray(scores).reshape(-1)

    if scores.min() < 0 or scores.max() > 1:
        probabilities = 1 / (1 + np.exp(-scores))
    else:
        probabilities = scores

    predictions = (probabilities >= 0.5).astype(int)

    accuracy = accuracy_score(val_true, predictions)
    macro_f1 = f1_score(
        val_true,
        predictions,
        average="macro"
    )

    results.append({
        "top_k": top_k,
        "accuracy": accuracy,
        "macro_f1": macro_f1
    })

    print(
        f"Top-{top_k} 결과 | "
        f"Accuracy: {accuracy:.4f} | "
        f"Macro-F1: {macro_f1:.4f}"
    )

    del model
    torch.cuda.empty_cache()
    gc.collect()

print("\n===== 최종 비교 =====")

for result in results:
    print(
        f"Top-{result['top_k']} | "
        f"Accuracy: {result['accuracy']:.4f} | "
        f"Macro-F1: {result['macro_f1']:.4f}"
    )

In [ ]:
def get_retrieved_titles(claim, top_k):
    row = retrieval_by_claim[claim]
    return row["doc_retrieval_results"][0][0][:top_k]


def normalized_title_set(titles):
    return {
        normalize_title(title)
        for title in titles
    }


def get_gold_titles(row):
    return {
        normalize_title(title)
        for title, sentence_id in row["supporting_facts"]
    }


def evaluate_doc_top_k(data, top_k):
    any_gold = 0
    all_gold = 0
    total_gold_docs = 0
    found_gold_docs = 0

    for row in data:
        retrieved = normalized_title_set(
            get_retrieved_titles(row["claim"], top_k)
        )

        gold = get_gold_titles(row)

        found = retrieved & gold

        if len(found) > 0:
            any_gold += 1

        if gold.issubset(retrieved):
            all_gold += 1

        found_gold_docs += len(found)
        total_gold_docs += len(gold)

    n = len(data)

    return {
        "top_k": top_k,
        "하나 이상 포함": any_gold / n,
        "정답 문서 전체 포함": all_gold / n,
        "정답 문서 Recall": found_gold_docs / total_gold_docs
    }


for k in [20, 30, 50, 100]:
    result = evaluate_doc_top_k(val_data, k)

    print(
        f"Top-{k:3d} | "
        f"하나 이상: {result['하나 이상 포함']:.3f} | "
        f"전체 포함: {result['정답 문서 전체 포함']:.3f} | "
        f"문서 Recall: {result['정답 문서 Recall']:.3f}"
    )

In [ ]:
from tqdm.auto import tqdm

DOC_TOP_K = 50

# 학습 + 검증 claim의 Top-50 문서 제목 수집
all_titles_top50 = set()
titles_by_claim_top50 = {}

for row in train_data + val_data:
    claim = row["claim"]

    titles = retrieval_by_claim[claim][
        "doc_retrieval_results"
    ][0][0][:DOC_TOP_K]

    titles_by_claim_top50[claim] = titles
    all_titles_top50.update(titles)

print("Top-50 고유 문서 수:", len(all_titles_top50))
print("기존 처리 문서 수:", len(candidate_sentences))

# 기존 Top-20 처리 문서는 제외
new_titles = [
    title
    for title in all_titles_top50
    if title not in candidate_sentences
]

print("새로 처리할 문서 수:", len(new_titles))

In [ ]:
new_errors = []

for title in tqdm(
    new_titles,
    desc="Top-50 추가 문서 문장 분리"
):
    try:
        sentences = get_sentences_corenlp(title)
        candidate_sentences[title] = tuple(sentences)
    except Exception as e:
        new_errors.append((title, str(e)))

print("새 문장 분리 완료 문서 수:", len(new_titles) - len(new_errors))
print("처리 오류 수:", len(new_errors))

if new_errors:
    print("오류 예시:")
    for error in new_errors[:5]:
        print(error)

In [ ]:
def get_gold_facts(row):
    return {
        (normalize_title(title), int(sentence_id))
        for title, sentence_id in row["supporting_facts"]
    }


def get_candidate_facts(row):
    facts = set()

    for title in titles_by_claim_top50[row["claim"]]:
        sentences = candidate_sentences.get(title, [])

        for sentence_id in range(len(sentences)):
            facts.add(
                (normalize_title(title), sentence_id)
            )

    return facts


gold_total = 0
found_total = 0
any_found = 0
all_found = 0

for row in tqdm(val_data, desc="Top-50 문장 후보 coverage"):
    gold = get_gold_facts(row)
    candidates = get_candidate_facts(row)

    found = gold & candidates

    gold_total += len(gold)
    found_total += len(found)

    if len(found) > 0:
        any_found += 1

    if gold.issubset(candidates):
        all_found += 1

print("\nTop-50 문장 후보 결과")
print("정답 문장 전체:", gold_total)
print("후보에 포함된 정답 문장:", found_total)
print("문장 Recall 상한:", found_total / gold_total)
print(
    "정답 문장 하나 이상 포함:",
    any_found / len(val_data)
)
print(
    "정답 문장 전체 포함:",
    all_found / len(val_data)
)

In [ ]:
val_flat_top50 = []

for row in tqdm(
    val_data,
    desc="Top-50 검증 문장 후보 생성"
):
    uid = row["uid"]
    claim = row["claim"]

    titles = titles_by_claim_top50[claim]

    for title in titles:
        sentences = candidate_sentences.get(title, [])

        for sentence_id, sentence in enumerate(sentences):
            val_flat_top50.append(
                (
                    uid,
                    {
                        "uid": uid,
                        "claim": claim,
                        "title": title,
                        "sentence_id": sentence_id,
                        "sentence": sentence
                    }
                )
            )

print("Top-50 검증 문장 후보 수:", len(val_flat_top50))

In [ ]:
val_top50_pairs = [
    (candidate["claim"], candidate["sentence"])
    for uid, candidate in val_flat_top50
]

val_top50_scores = []

for start in tqdm(
    range(0, len(val_top50_pairs), 10000),
    desc="Top-50 Cross-Encoder 점수 계산"
):
    batch_pairs = val_top50_pairs[start:start + 10000]

    scores = base_encoder.predict(
        batch_pairs,
        batch_size=128,
        show_progress_bar=False
    )

    val_top50_scores.extend(
        np.asarray(scores).reshape(-1)
    )

val_top50_scores = np.asarray(val_top50_scores)

print("계산된 점수 수:", len(val_top50_scores))

In [ ]:
from collections import defaultdict

ranked_val_top50 = defaultdict(list)

for (uid, candidate), score in zip(
    val_flat_top50,
    val_top50_scores
):
    ranked_val_top50[uid].append({
        "title": candidate["title"],
        "sentence_id": candidate["sentence_id"],
        "sentence": candidate["sentence"],
        "score": float(score)
    })

gold_by_uid_top50 = {
    row["uid"]: get_gold_facts(row)
    for row in val_data
}

def fact_key(item):
    return (
        normalize_title(item["title"]),
        int(item["sentence_id"])
    )


def evaluate_top50_ranking(top_k):
    total_gold = 0
    found_gold = 0
    claims_with_any = 0
    claims_with_all = 0

    for uid, ranked_items in ranked_val_top50.items():
        ranked_items = sorted(
            ranked_items,
            key=lambda x: x["score"],
            reverse=True
        )

        selected = ranked_items[:top_k]
        selected_facts = {
            fact_key(item)
            for item in selected
        }

        gold_facts = gold_by_uid_top50[uid]
        found = selected_facts & gold_facts

        total_gold += len(gold_facts)
        found_gold += len(found)

        if found:
            claims_with_any += 1

        if gold_facts.issubset(selected_facts):
            claims_with_all += 1

    n_claims = len(ranked_val_top50)

    return {
        "recall": found_gold / total_gold,
        "any": claims_with_any / n_claims,
        "all": claims_with_all / n_claims
    }


print("\n[Top-50 + 사전학습 Cross-Encoder]")

for k in [1, 3, 5, 10]:
    result = evaluate_top50_ranking(k)

    print(
        f"Top-{k:2d} | "
        f"근거 Recall: {result['recall']:.3f} | "
        f"하나 이상: {result['any']:.3f} | "
        f"전체 포함: {result['all']:.3f}"
    )

In [ ]:
train_flat_top50 = []

for row in tqdm(
    train_data,
    desc="Top-50 학습 문장 후보 생성"
):
    claim = row["claim"]

    for title in titles_by_claim_top50[claim]:
        sentences = candidate_sentences.get(title, [])

        for sentence_id, sentence in enumerate(sentences):
            train_flat_top50.append(
                (
                    row["uid"],
                    {
                        "uid": row["uid"],
                        "claim": claim,
                        "title": title,
                        "sentence_id": sentence_id,
                        "sentence": sentence
                    }
                )
            )

print("Top-50 학습 문장 후보 수:", len(train_flat_top50))

In [ ]:
train_top50_pairs = [
    (candidate["claim"], candidate["sentence"])
    for uid, candidate in train_flat_top50
]

train_top50_scores = []

for start in tqdm(
    range(0, len(train_top50_pairs), 10000),
    desc="Top-50 학습 점수 계산"
):
    batch_pairs = train_top50_pairs[start:start + 10000]

    scores = base_encoder.predict(
        batch_pairs,
        batch_size=128,
        show_progress_bar=False
    )

    train_top50_scores.extend(
        np.asarray(scores).reshape(-1)
    )

train_top50_scores = np.asarray(train_top50_scores)

print("계산된 학습 점수 수:", len(train_top50_scores))

In [ ]:
ranked_train_top50 = defaultdict(list)

for (uid, candidate), score in zip(
    train_flat_top50,
    train_top50_scores
):
    ranked_train_top50[uid].append({
        "title": candidate["title"],
        "sentence_id": candidate["sentence_id"],
        "sentence": candidate["sentence"],
        "score": float(score)
    })

label_by_uid = {
    row["uid"]: row["label"]
    for row in train_data
}

train_top50_verifier_inputs = {}

for uid, candidates in ranked_train_top50.items():
    ranked = sorted(
        candidates,
        key=lambda x: x["score"],
        reverse=True
    )[:5]

    evidence_text = "\n".join(
        f"Evidence {i+1} [{item['title']}, 문장 {item['sentence_id']}]: "
        f"{item['sentence']}"
        for i, item in enumerate(ranked)
    )

    claim = next(
        candidate["claim"]
        for uid2, candidate in train_flat_top50
        if uid2 == uid
    )

    train_top50_verifier_inputs[uid] = {
        "claim": claim,
        "evidence_text": evidence_text,
        "ranked_evidence": ranked,
        "label": label_by_uid[uid]
    }

print(
    "Top-50 기반 최종 판정 학습 입력:",
    len(train_top50_verifier_inputs)
)

In [ ]:
val_claim_by_uid = {
    row["uid"]: row["claim"]
    for row in val_data
}

val_label_by_uid = {
    row["uid"]: row["label"]
    for row in val_data
}

val_top50_verifier_inputs = {}

for uid, candidates in ranked_val_top50.items():
    ranked = sorted(
        candidates,
        key=lambda x: x["score"],
        reverse=True
    )[:5]

    evidence_text = "\n".join(
        f"Evidence {i+1} [{item['title']}, 문장 {item['sentence_id']}]: "
        f"{item['sentence']}"
        for i, item in enumerate(ranked)
    )

    val_top50_verifier_inputs[uid] = {
        "claim": val_claim_by_uid[uid],
        "evidence_text": evidence_text,
        "ranked_evidence": ranked,
        "label": val_label_by_uid[uid]
    }

print(
    "Top-50 기반 검증 입력:",
    len(val_top50_verifier_inputs)
)

In [ ]:
import random
from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder, InputExample
from sklearn.utils import resample

train_items_top50 = list(
    train_top50_verifier_inputs.values()
)

supported_items = [
    item for item in train_items_top50
    if item["label"] == "SUPPORTED"
]

not_supported_items = [
    item for item in train_items_top50
    if item["label"] == "NOT_SUPPORTED"
]

# NOT_SUPPORTED oversampling
not_supported_balanced = resample(
    not_supported_items,
    replace=True,
    n_samples=len(supported_items),
    random_state=42
)

balanced_items = (
    supported_items +
    not_supported_balanced
)

random.Random(42).shuffle(balanced_items)

train_examples_top50 = [
    InputExample(
        texts=[
            item["claim"],
            item["evidence_text"]
        ],
        label=1.0 if item["label"] == "SUPPORTED" else 0.0
    )
    for item in balanced_items
]

train_loader_top50 = DataLoader(
    train_examples_top50,
    shuffle=True,
    batch_size=16
)

print("Top-50 균형 학습 쌍:", len(train_examples_top50))

In [ ]:
final_verifier_top50 = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    num_labels=1,
    max_length=512,
    device="cuda"
)

final_verifier_top50.fit(
    train_dataloader=train_loader_top50,
    epochs=1,
    warmup_steps=int(len(train_loader_top50) * 0.1),
    optimizer_params={"lr": 2e-5},
    output_path="/content/hover_final_verifier_top50",
    use_amp=True,
    show_progress_bar=True
)

print("Top-50 최종 판정 모델 학습 완료")

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

val_items_top50 = list(
    val_top50_verifier_inputs.values()
)

val_pairs_top50 = [
    (
        item["claim"],
        item["evidence_text"]
    )
    for item in val_items_top50
]

val_true_top50 = np.array([
    1 if item["label"] == "SUPPORTED" else 0
    for item in val_items_top50
])

scores_top50 = final_verifier_top50.predict(
    val_pairs_top50,
    batch_size=32,
    show_progress_bar=True
)

scores_top50 = np.asarray(scores_top50).reshape(-1)

if scores_top50.min() < 0 or scores_top50.max() > 1:
    probabilities_top50 = 1 / (1 + np.exp(-scores_top50))
else:
    probabilities_top50 = scores_top50

predictions_top50 = (
    probabilities_top50 >= 0.5
).astype(int)

print(
    "Accuracy:",
    accuracy_score(val_true_top50, predictions_top50)
)

print(
    "Macro-F1:",
    f1_score(
        val_true_top50,
        predictions_top50,
        average="macro"
    )
)

print("\n라벨별 결과:")
print(
    classification_report(
        val_true_top50,
        predictions_top50,
        target_names=[
            "NOT_SUPPORTED",
            "SUPPORTED"
        ]
    )
)

print("\n혼동 행렬:")
print(
    confusion_matrix(
        val_true_top50,
        predictions_top50
    )
)

In [ ]:
from collections import defaultdict
import random
import numpy as np
from tqdm.auto import tqdm

# 학습 claim별 정답 근거
gold_train_by_uid = {
    row["uid"]: {
        (normalize_title(title), int(sentence_id))
        for title, sentence_id in row["supporting_facts"]
    }
    for row in train_data
}

# 학습 후보를 claim별로 묶기
train_grouped_top50 = defaultdict(list)

for (uid, candidate), score in zip(
    train_flat_top50,
    train_top50_scores
):
    train_grouped_top50[uid].append({
        "claim": candidate["claim"],
        "title": candidate["title"],
        "sentence_id": candidate["sentence_id"],
        "sentence": candidate["sentence"],
        "score": float(score)
    })

hard_pairs = []

for uid, candidates in tqdm(
    train_grouped_top50.items(),
    desc="Hard negative 생성"
):
    gold_facts = gold_train_by_uid[uid]

    positives = []
    negatives = []

    for candidate in candidates:
        fact = (
            normalize_title(candidate["title"]),
            int(candidate["sentence_id"])
        )

        if fact in gold_facts:
            positives.append(candidate)
        else:
            negatives.append(candidate)

    # 정답 문장이 후보에 없는 claim은 reranker 학습에서 제외
    if not positives:
        continue

    # 기존 모델 점수가 높은 오답만 hard negative로 선택
    negatives = sorted(
        negatives,
        key=lambda x: x["score"],
        reverse=True
    )[:10]

    # 정답 문장 추가
    for candidate in positives:
        hard_pairs.append({
            "claim": candidate["claim"],
            "sentence": candidate["sentence"],
            "label": 1
        })

    # 어려운 오답 문장 추가
    for candidate in negatives:
        hard_pairs.append({
            "claim": candidate["claim"],
            "sentence": candidate["sentence"],
            "label": 0
        })

print("Hard negative 전체 쌍:", len(hard_pairs))

positive_pairs = [
    item for item in hard_pairs
    if item["label"] == 1
]

negative_pairs = [
    item for item in hard_pairs
    if item["label"] == 0
]

print("Positive:", len(positive_pairs))
print("Negative:", len(negative_pairs))

In [ ]:
from sklearn.utils import resample

negative_balanced = resample(
    negative_pairs,
    replace=True,
    n_samples=len(positive_pairs),
    random_state=42
)

balanced_hard_pairs = (
    positive_pairs + negative_balanced
)

random.Random(42).shuffle(balanced_hard_pairs)

print(
    "균형 조정 후 hard negative 학습 쌍:",
    len(balanced_hard_pairs)
)

In [ ]:
from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder, InputExample

hard_train_examples = [
    InputExample(
        texts=[
            item["claim"],
            item["sentence"]
        ],
        label=float(item["label"])
    )
    for item in balanced_hard_pairs
]

hard_train_loader = DataLoader(
    hard_train_examples,
    shuffle=True,
    batch_size=32
)

hard_negative_reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    num_labels=1,
    max_length=256,
    device="cuda"
)

hard_negative_reranker.fit(
    train_dataloader=hard_train_loader,
    epochs=1,
    warmup_steps=int(len(hard_train_loader) * 0.1),
    optimizer_params={"lr": 1e-5},
    output_path="/content/hover_hard_negative_reranker",
    use_amp=True,
    show_progress_bar=True
)

print("Hard negative reranker 학습 완료")

In [ ]:
val_hard_scores = []

for start in tqdm(
    range(0, len(val_top50_pairs), 10000),
    desc="Hard negative reranker 검증 점수"
):
    batch_pairs = val_top50_pairs[start:start + 10000]

    scores = hard_negative_reranker.predict(
        batch_pairs,
        batch_size=128,
        show_progress_bar=False
    )

    val_hard_scores.extend(
        np.asarray(scores).reshape(-1)
    )

val_hard_scores = np.asarray(val_hard_scores)

print("검증 점수 수:", len(val_hard_scores))

In [ ]:
ranked_val_hard = defaultdict(list)

for (uid, candidate), score in zip(
    val_flat_top50,
    val_hard_scores
):
    ranked_val_hard[uid].append({
        "title": candidate["title"],
        "sentence_id": candidate["sentence_id"],
        "sentence": candidate["sentence"],
        "score": float(score)
    })


def evaluate_hard_reranker(top_k):
    total_gold = 0
    found_gold = 0
    any_found = 0
    all_found = 0

    for uid, candidates in ranked_val_hard.items():
        ranked = sorted(
            candidates,
            key=lambda x: x["score"],
            reverse=True
        )

        selected = ranked[:top_k]

        selected_facts = {
            (
                normalize_title(item["title"]),
                int(item["sentence_id"])
            )
            for item in selected
        }

        gold_facts = gold_by_uid_top50[uid]
        found = selected_facts & gold_facts

        total_gold += len(gold_facts)
        found_gold += len(found)

        if found:
            any_found += 1

        if gold_facts.issubset(selected_facts):
            all_found += 1

    n = len(ranked_val_hard)

    return {
        "recall": found_gold / total_gold,
        "any": any_found / n,
        "all": all_found / n
    }


print("\n[Hard negative reranker 결과]")

for k in [1, 3, 5, 10]:
    result = evaluate_hard_reranker(k)

    print(
        f"Top-{k:2d} | "
        f"근거 Recall: {result['recall']:.3f} | "
        f"하나 이상: {result['any']:.3f} | "
        f"전체 포함: {result['all']:.3f}"
    )

In [ ]:
def evaluate_score_array(scores, name):
    ranked = defaultdict(list)

    for (uid, candidate), score in zip(
        val_flat_top50,
        scores
    ):
        ranked[uid].append({
            "title": candidate["title"],
            "sentence_id": candidate["sentence_id"],
            "score": float(score)
        })

    total_gold = 0
    found_gold = 0

    for uid, candidates in ranked.items():
        selected = sorted(
            candidates,
            key=lambda x: x["score"],
            reverse=True
        )[:10]

        selected_facts = {
            (
                normalize_title(item["title"]),
                int(item["sentence_id"])
            )
            for item in selected
        }

        gold = gold_by_uid_top50[uid]

        total_gold += len(gold)
        found_gold += len(selected_facts & gold)

    print(
        name,
        "Top-10 Recall:",
        round(found_gold / total_gold, 4)
    )


evaluate_score_array(
    val_top50_scores,
    "사전학습 Cross-Encoder"
)

evaluate_score_array(
    val_hard_scores,
    "Hard negative 원래 점수"
)

evaluate_score_array(
    -val_hard_scores,
    "Hard negative 점수 반전"
)

In [ ]:
from functools import lru_cache
from collections import defaultdict
import numpy as np

allowed_entity_labels = {
    "PERSON",
    "ORG",
    "GPE",
    "LOC",
    "FAC",
    "EVENT",
    "WORK_OF_ART"
}


@lru_cache(maxsize=10000)
def lookup_entity_titles(entity, limit=20):
    entity = entity.strip()

    if not entity:
        return []

    # 제목 정확히 일치
    rows = conn.execute(
        "SELECT id FROM documents WHERE id = ? LIMIT ?",
        (entity, limit)
    ).fetchall()

    if rows:
        return [row[0] for row in rows]

    # 제목이 해당 이름으로 시작하는 문서 검색
    escaped = (
        entity
        .replace("*", "[*]")
        .replace("?", "[?]")
    )

    rows = conn.execute(
        "SELECT id FROM documents "
        "WHERE id GLOB ? "
        "LIMIT ?",
        (escaped + "*", limit)
    ).fetchall()

    return [row[0] for row in rows]


def get_title_sentences(title):
    # 이미 문장 분리된 문서는 캐시 사용
    if title in candidate_sentences:
        return candidate_sentences[title]

    # 새 문서는 CoreNLP로 처리
    sentences = get_sentences_corenlp(title)
    candidate_sentences[title] = tuple(sentences)

    return candidate_sentences[title]


def score_sentence_pairs(pairs):
    if not pairs:
        return np.array([])

    scores = base_encoder.predict(
        pairs,
        batch_size=128,
        show_progress_bar=False
    )

    return np.asarray(scores).reshape(-1)